<a href="https://colab.research.google.com/github/allenswdb/TReND-CaMinA/blob/main/notebooks/Kenya26/03_04-Wed1toThu2-AllenTutorial/project_templates/Project-8_Classifying-Transgenic-Lines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exploring the neural coding & behavioral dimensions of different transgenic lines

---

### Background

The **Allen Brain Observatory** is a large-scale public dataset of two-photon calcium imaging (optical physiology) collected from head-fixed mice passively viewing visual stimuli. A key feature of the dataset is its use of **transgenic Cre lines**: each mouse expresses a fluorescent calcium indicator (typically GCaMP6f) under the control of a Cre recombinase whose expression is restricted to a specific neuronal cell type (e.g., layer 2/3 excitatory neurons, layer 5 excitatory neurons, SST+ interneurons). All animals share the same inbred background strain (**C57BL/6J**), differ only in which Cre allele they carry.

During each imaging session (~60 min), mice are free to run on a treadmill and are tracked by an eye-tracking camera, yielding continuous **running speed** and **pupil size / gaze position** time series alongside the neural fluorescence data. How these behavioral variables relate to neural activity has been extensively quantified in *de Vries 2020* and consolidated into a cell metrics table for future analysis. This tutorial will utilize this table & pre-computed behavioral variables to create an project space independent of the full dataset, which is a behemoth of nearly 1,400 experiments!!

### Motivation

The cre lines used in this study were designed to investigate different populations of neurons, so we should expect differences in visual coding properties. We will investigate this using a pre-computed metrics table to expand our understanding in the various dimensions explored in these experiments. Namely, stimulus driven activity & how this activity relates to the behavior of the animal. To quantify this, decoding will be utilized to determine which transgenic lines can be classified from each other and with which features. If this is successful, we can further ask, can we use behavior by itself to classify cre lines from each other. Because all Cre lines were derived from the same background strain, the naïve expectation is that behavioral phenotypes should be essentially identical across lines.

### Project Goals

1. **Identify neural coding dimensions** that vary across cortical layers, cortical areas, and transgenic lines
2. **Characterize running and pupil behavior** across all transgenic lines in the Allen Brain Observatory optical physiology dataset.
3. **Train and evaluate linear classifiers** (SVM, LDA, Logistic Regression) to decode Cre-line identity from neural coding metrics & behavioral features alone, using stratified cross-validation.
4. **Validate against a shuffle-label null distribution** to rigorously test whether decoding accuracy exceeds chance.
5. **Identify the most discriminative features** and the most confusable pairs of transgenic lines.


In [ ]:
# @title Run to initialize Allen Brain Observatory on Colab {display-mode: "form" }

# run only once per runtime/session, and only if running in colab
# the runtime will need to restxart after
%%capture
!apt install s3fs

!pip uninstall -y numpy pandas
!pip install git+https://github.com/AllenInstitute/AllenSDK@1bdca3ad884c3a5edea8236161424650603e6f29 "numpy == 1.26.4" "pandas == 2.3.0" "matplotlib > 3.8.0" "statsmodels >= 0.14.4"
import allensdk
print('allensdk imported successfully')

!mkdir -p /data/allen-brain-observatory/
!s3fs allen-brain-observatory /data/allen-brain-observatory/ -o public_bucket=1

import time
print("Runtime is now restarting...")
print("You can ignore the error message [Your session crashed for an unknown reason.]")
time.sleep(5)
exit()

In [ ]:
%matplotlib inline
## Import Required Libraries
#Base
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as st

#Core
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

#Plot
import matplotlib.pyplot as plt
import seaborn as sns

#Live dangerously
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import platform, os, sys
platstring = platform.platform()

if ('amzn' in platstring) or ('google.colab' in sys.modules):
    # for AWS
    vc_cache_dir = '/data/allen-brain-observatory/visual-coding-2p'
else:  
    # for local drive, different operating systems
    if ('Darwin' in platstring) or ('macOS' in platstring):
        # OS X 
        data_root = "/Volumes/TReND2026/"
    elif 'Windows'  in platstring:
        # Windows (replace with the drive letter of USB drive)
        data_root = "E:/"
    else:
        # your own linux platform
        # EDIT location where you mounted hard drive
        data_root = "/media/$USERNAME/TReND2026/"
        
    # visual behavior cache directory
    vc_cache_dir = os.path.join(data_root, "visual_coding_2p")
    
boc = BrainObservatoryCache(manifest_file=os.path.join(vc_cache_dir, 'manifest.json'))

In [ ]:
#Data
DATA_DIR = Path("/data/visual_coding_ophys")
MANIFEST_PATH = DATA_DIR / "brain_observatory_manifest.json"
NWB_DIR = DATA_DIR / "ophys_experiment_data"

session_types = ['three_session_A','three_session_B','three_session_C','three_session_C2']
# Initialize the cache. Setting `manifest_file` to the local manifest avoids any download.
boc = BrainObservatoryCache(manifest_file=str(MANIFEST_PATH))

# Get high-level metadata tables
containers_df = pd.DataFrame(boc.get_experiment_containers())
containers_df = containers_df.rename(columns={'id':'experiment_container_id'}).set_index('experiment_container_id')
experiments_df = pd.DataFrame(boc.get_ophys_experiments())
experiments_df = experiments_df[experiments_df['session_type'].isin(session_types)]
uniq_container_ids = experiments_df['experiment_container_id'].unique()
containers_df = containers_df.loc[uniq_container_ids]
experiments_df.set_index('id', inplace=True)

print(f"{len(containers_df)} experiment containers, {len(experiments_df)} ophys experiments")
experiments_df.head()

# 1. How do visual responses of excitatory cells differ across cortical depths using transgenic lines?

In [ ]:
#Read in pandas dataframe with pre-computed metrics for all cells
metrics_all = pd.read_csv("metrics_clean_2025.csv",index_col=0)

uniq_cre_lines = np.unique(metrics_all['tld1_name'])
uniq_areas = np.unique(metrics_all['area'])
nCre_lines = len(uniq_cre_lines)
nAreas = len(uniq_areas)
print(f'Unique areas recorded: {uniq_areas}')
print(f"Metrics table shape: {metrics_all.shape[0]} cells across {nCre_lines} cre lines and {nAreas} areas with {metrics_all.shape[1]} columns !! Woah that is too much to look at! ")
metrics_all.head()

In [ ]:
## Subsample the metrics table to focus on a subset of cre lines and one area (VISp) for this analysis
# Below are the 8 lines we will focus on for this analysis, which have more specific expression patterns in different cortical layers and cell types
# Excluding 'Emx1-IRES-Cre', 'Slc17a7-IRES2-Cre' because they are pan-excitatory lines and we want to focus on the more specific lines that target particular cortical layers and cell types for this analysis.
#
# What do all of these transgenic lines (tld1_name) even mean?
#       Look here -> https://observatory.brain-map.org/visualcoding/transgenic
#
# Don't panic! Most computational neuroscientists (myself included) don't understand the full details of all of these transgenic lines, how they are created, etc,
# but the key point is that they allow us to target different subsets of cells across the brain, which is very useful for understanding how diverse cell types / brain areas contribute to visual processing and behavior.

cre_lines_of_interest = [
                        'Cux2-CreERT2',     #GCaMP6f expression is enriched in cortical layers 2, 3 and 4, among other areas
                        'Rorb-IRES2-Cre',   #GCaMP6f expression is predominantly in excitatory neurons in cortical layer 4 (dense) and layers 5/6 (sparse), among other areas
                        'Scnn1a-Tg3-Cre',   #GCaMP6f is predominantly expressed in excitatory neurons in cortical layer 4, among other areas
                        'Nr5a1-Cre',        #GCaMP6f is predominantly expressed in excitatory neurons in cortical layer 4 , among other areas
                        'Fezf2-CreER',      #GCaMP6f is strongly expressed in cortical layer 5 and some layer 6 excitatory neurons, among other areas
                        'Tlx3-Cre_PL56',    #GCaMP6f is strongly expressed in cortical layer 5 and some layer 6 excitatory neurons, among other areas
                        'Rbp4-Cre_KL100',   #GCaMP6f is predominantly expressed in excitatory neurons in cortical layer 5, among other areas
                        'Ntsr1-Cre_GN220',  #GCaMP6f expression is specific to cortical layer 6 neurons
                        ]

#Pandas query allows us to subset the metrics_all dataframe to only include rows where area is VISp and tld1_name is in our list of cre lines of interest
subsample_criteria = f"(area == 'VISp') & (tld1_name.isin(@cre_lines_of_interest))"
metrics_VISp = metrics_all.query(subsample_criteria)
print(f"After subsampling to focus on VISp and {len(cre_lines_of_interest)} cre lines of interest labeling specific populations of excitatory cells, we *only* have {metrics_VISp.shape[0]} cells")

In [ ]:
## Since we have so many pre-computed metrics, here is a brief summary of some of the most important columns
# More details can be obtained from the platform paper  (de Vries et al., 2020) & from the TAs
#
# _dg suffix means the metric was computed for the drifting grating stimulus responses
# _sg suffix means the metric was computed for the static grating stimulus responses
# _nm3 suffix means the metric was computed for the natural movie 3 stimulus responses
# _nm1 suffix means the metric was computed for the natural movie 1 stimulus responses
# _ns suffix means the metric was computed for natural scene stimulus responses
#
# DRIFTING GRATINGS METRICS:
# 'pref_ori_dg':            preferred orientation for drifting gratings
# 'pref_tf_dg':             preferred temporal frequency for drifting gratings
# 'num_pref_trials_dg':     number of trials at the preferred orientation for drifting gratings
# 'responsive_dg':          whether the cell is responsive to drifting gratings
# 'g_osi_dg':               orientation selectivity index for drifting gratings
# 'g_dsi_dg':               direction selectivity index for drifting gratings
# 'tfdi_dg':                temporal frequency direction index for drifting gratings
# 'reliability_dg':         defined by the percentage of significant responses to repeated presentations of drifting gratings
# 'lifetime_sparseness_dg': lifetime sparseness of the cell's response to drifting gratings
# 'cv_dg':                  coefficient of variation of the cell's response to drifting gratings
#
# RUNNING METRICS:
# 'run_pval_dg':            p-value for running modulation for drifting gratings
# 'run_resp_dg':            running response for drifting gratings
# 'run_mod_dg':             running modulation for drifting gratings
# 'responsive_rundg':       whether the cell is responsive to drifting gratings during running
# 'run_corr_mean':          mean correlation of the cell's response to running speed across all sessions
# 'run_corr_A_lw':          mean correlation of the cell's response to running speed during session A (drifting gratings & natural movies)
# 'run_corr_B_lw':          mean correlation of the cell's response to running speed during session B (static gratings & natural scenes)
# 'run_corr_C_lw':          mean correlation of the cell's response to running speed during session C (locally sparse noise & natural movies)
# 'responsive_run':         whether the cell is responsive to running


In [ ]:
# Drifting gratings is a commonly used visual stimulus in systems neuroscience, so let's start by looking at some of these metrics for different transgenic lines.
# 24 columns is much more manageable! 
metrics_dg = metrics_VISp[['cell_specimen_id','experiment_container_id','tld1_name','area','imaging_depth','depth_range',
                          'pref_ori_dg','pref_tf_dg','num_pref_trials_dg','responsive_dg','g_osi_dg',
                          'g_dsi_dg','tfdi_dg','reliability_dg','lifetime_sparseness_dg','cv_dg',
                          'run_pval_dg','run_resp_dg','run_mod_dg','responsive_rundg','run_corr_mean',
                          'run_corr_A_lw','run_corr_B_lw','run_corr_C_lw','responsive_run']]
metrics_dg = metrics_dg.set_index('cell_specimen_id')
metrics_dg.head()

In [ ]:
## How many cells have pre-computed metrics for across different cortical layers?
# Ignoring for a moment the complexity of different transgenic lines, let's look at how many cells we have with pre-computed metrics across different cortical layers (depth_range column) in VISp.
#
# 100: L2/3
# 200: L4
# 300: L5
# 500: L6
depth_ranges = np.array([100, 200, 300, 500])

#To make this more interpretable, let's map the depth_range values to their corresponding layer labels
layer_labels = ['L2/3', 'L4', 'L5', 'L6']
metrics_dg['layer'] = metrics_dg['depth_range'].map(dict(zip(depth_ranges, layer_labels)))

#If you're interested in decoding layer & area labels, you can also create a new column that combines area and layer information
metrics_dg['area_layer'] = metrics_dg['area'] + '_' + metrics_dg['layer']

#How many cells do we have with pre-computed metrics across different layers?
print("Number of cells with pre-computed metrics across different layers in VISp:")
metrics_dg['layer'].value_counts().sort_index()


In [ ]:
#Calculate the proportion of units that are responsive to drifting gratings for each tld1 line
#Normalize by the total number of units recorded for each line to get a proportion
proportion_units_responsive_dg = metrics_dg[["layer", "responsive_dg"]].groupby("layer").sum()/metrics_dg[["layer", "responsive_dg"]].groupby("layer").count()
proportion_units_responsive_dg

In [ ]:
# Looking at this table, we can see that deeper cortical layers have a higher proportion of cells that are responsive to drifting gratings than superficial L2/3.
# Let's plot the distribution of reliability (quantified as the % of significant responses) to drifting gratings of these 2 depths

def plot_metric_distributions(metric_col, group_col='layer', binwidth=None, group_order=layer_labels, title=None, test_significance=False):

    cmap = sns.color_palette("Set1", n_colors=len(group_order))
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_title(title)
    sns.histplot(data=metrics_dg[metrics_dg[group_col].isin(group_order)], x=metric_col, hue=group_col,palette=cmap,hue_order=group_order, multiple='layer',element='step',stat='density',common_norm=False, binwidth=binwidth, ax=ax)

    #Check significance between 2 distributions
    if (test_significance) and (len(group_order) == 2):
        distr_g1 = metrics_dg.loc[metrics_dg[group_col] == group_order[0],metric_col].values
        distr_g2 =  metrics_dg.loc[metrics_dg[group_col] == group_order[1],metric_col].values
        res = st.mannwhitneyu(distr_g1,distr_g2,nan_policy='omit')

        if res.pvalue < 0.05:
            ax.text(0.5,0.925,'*',transform=ax.transAxes,fontsize=22,fontweight='bold')

title_str = "L5 cells have higher reliability to drifting gratings than L2/3 cells"
plot_metric_distributions('reliability_dg', group_order=['L2/3','L5'],title=title_str,binwidth=0.025, test_significance=True)


In [ ]:
# PLOTTING EXERCISE: We found that L5 cells have a higher proportion of responsive cells to drifting gratings than L2/3 cells, and that L5 cells also have higher reliability to drifting gratings than L2/3 cells.
# What do the other 2 layers look like? Does this pattern hold across all layers? Can you find 2 layers where the opposite is true? 
# (i.e. one line has a higher proportion of responsive cells, but lower reliability than the other line)


In [ ]:
## Exercise: As a corrollary to reliability, what does the sparseness of the responses to drifting gratings look like across different cortical depths? 
# Sparseness is a measure of how selective a neuron is to different stimuli. A neuron that responds strongly to only a few stimuli will have a lifetime sparseness close to 1, 
# whereas a neuron that responds broadly to many stimuli will  have a lower lifetime sparseness.


In [ ]:
## EXERCISE: Using the same pandas code as above, Calculate the proportion of units that are responsive to locomotion in general ("responsive_run") & locomotion during drifting_gratings ("responsive_rundg") for each layer
# Which layer has the highest proportion of responsive cells to locomotion? 


In [ ]:
## Different cortical depths have different proportions of cells that are modulated by running, which is interesting to think 
# about in the context of how the neurons embedding in a neural circuit contribute to visual processing during behavior. 
#
# PLOTTING EXERCISE
# Let's plot the distribution of running modulation during drifting gratings ("run_mod_dg") for superficial (L2/3) vs deep (L5) layers, 
# since we found that they have different proportions of responsive cells to drifting gratings and different reliability to drifting gratings, which might be related to how modulated they are by running during drifting gratings.


In [ ]:
# CONTINUING EXERCISES
# Explore how different metrics vary across different cortical depths or even brain regions
# -> -> Metrics like direction selectivity ('g_dsi_dg'), orientation selectivity ('g_osi_dg'), lifetime sparseness ('lifetime_sparseness_dg'), coefficient of variation ('cv_dg') are all interesting metrics to explore across different layers and brain areas.
# Try to leverage Pandas & Seaborn functionalities to explore these relationships and visualize them in a way that is easy to interpret.
# Perhaps start with "run_corr_mean", or look at Fig 6. of the platform paper (de Vries et al., 2020) to get inspiration 
# If you are brave, you can even look at different stimuli (e.g. static gratings, natural movies, etc) and see how these relationships change across different stimulus types.

In [ ]:
for metric in ['g_osi_dg', 'g_dsi_dg','lifetime_sparseness_dg','cv_dg']:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.set_title(f"{metric}")
    sns.boxplot(data=metrics_dg, y=metric, x='layer',order=layer_labels, ax=ax,showfliers=False)

# 2. Are these neural coding dimensions predictive of cortical depth?

In [ ]:
## Project motivation & possible directions
# Above we were looking at metrics over all cells recorded in V1 in a subset of excitatory cre lines,
# but we can also look at different metrics on a single experiment level and ask if that is predictive
# of the type of transgenic animal the recording was taken from.
# We will have to decide what features our classifier will use and what decoding technique to try
# I will set up the features using drifting grating metrics below, but feel free to get creative and try different approaches!


In [ ]:
min_units = 10
summary_per_experiment = []
#Loop over unique container ids (all imaging sessions for 1 imaging plane; i.e. some cells might have been recorded across multiple sessions)
for exp_c_id in np.unique(metrics_dg['experiment_container_id']):
    exp_data = metrics_dg[metrics_dg['experiment_container_id'] == exp_c_id]
    n_units = len(exp_data)

    if n_units < min_units:
        continue

    #Drifting grating metrics
    summary_per_experiment.append({

        #Metadata about the experiment
        'layer': exp_data['layer'].values[0],
        'experiment_container_id': exp_c_id,
        'area': containers_df.at[exp_c_id, 'targeted_structure'],
        'cre_line': containers_df.at[exp_c_id, 'cre_line'],
        'imaging_depth': containers_df.at[exp_c_id, 'imaging_depth'],
        'n_units': n_units,

        # Proportion of responsive cells to drifting gratings for this experiment
        'resp_dg_prop': exp_data['responsive_dg'].sum() / n_units,
        
        #Here we will calculate the mean of each drifting grating metric across all cells in the experiment,
        #but you could think about calculating other summary statistics like min, max, std, etc,
        #or calculating these metrics across only the responsive cells in the experiment, etc
        'reliability': np.nanmean(exp_data['reliability_dg']),
        'sparseness': np.nanmean(exp_data['lifetime_sparseness_dg']),
        'osi': np.nanmean(exp_data['g_osi_dg']),
        'dsi': np.nanmean(exp_data['g_dsi_dg']),
        'cv': np.nanmean(exp_data['cv_dg']),
    })

#You can create a pandas dataframe like this
summary_df = pd.DataFrame(summary_per_experiment)
summary_df.head()

In [ ]:
# --- Build feature matrix and labels ---
features = ['resp_dg_prop', 
            'reliability','sparseness', 'osi', 'dsi','cv',
            ]

target = 'layer'

# Drop rows with NaNs in any predictor or the label
clf_df = summary_df[features + [target]].dropna().copy()
X = clf_df[features].to_numpy()
y = clf_df[target].to_numpy()

print(f"Samples: {len(clf_df)}, features: {X.shape[1]}, classes: {np.unique(y).size}")

# How many data points do we have for each transgenic line?
summary_df['layer'].value_counts()


In [ ]:
## EXERCISE: Try to decode layer labels from these features using a simple classifier like LDA, SVM, or logistic regression.
# Use stratified cross-validation to evaluate the performance of your classifier, and look at the confusion matrix to see which layers are most often confused with each other.
# You can also try to decode other labels like area or cre_line, but keep in mind that you will have fewer data points for each class
# & we have the added complexity of remembering what each transgenic lines was designed for (e.g. for which layer)
#
#These are some useful libraries / functions for doing classification in Python, but feel free to use any other libraries you are comfortable with!
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# DECODE WHICH LAYER THE EXPERIMENT WAS RECORDED FROM USING YOUR CLASSIFIER OF CHOICE

In [ ]:
## EXERCISE: Can we think of a simple way to test if our decoding accuracy is significantly above chance by creating a shuffle distribution?
# The simplest way to do this is to shuffle the labels (layer) using np.random.permutation across experiments and re-run the same CV procedure. 
# This will break any real relationship between the features and the labels, and give us a null distribution of accuracies we can compare our real accuracy against.
n_shuffles_simple = 100



In [ ]:
## EXERCISE: Which features are most predictive in our decoder?
# For a *linear* model (linear SVM, LDA, LogReg), the magnitude of the fitted coefficient
# for each (standardized) feature is a direct measure of how strongly it contributes to the
# decision boundary. A more general-purpose approach that works for any pipeline is
# permutation importance: shuffle one feature at a time, see how much CV accuracy drops.
# or you could just look at the magnitude of the coefficients of your classifier (e.g. svm.coef_)
from sklearn.inspection import permutation_importance


## MORE PROJECT IDEAS & DIRECTIONS OF EXPLORATION
1. Choose different features to train your classifier on and see if you can decode cortical layers with different sets of features.
- Try features that relate neural activity to locomotion. Is this enough to classify cortical layers? ['resp_run_prop', 'resp_rundg_prop','run_mod', 'run_corr', 'run_mod_resp', 'run_corr_resp'] 
2. Instead of decoding cortical depth within a particular area, can you decode brain area or cell-type (by adding 'Vip-IRES-Cre' & 'Sst-IRES-Cre' back to the metrics table)?
- Combinatorially, this can get pretty complex, but you can start by trying to decode VISp, VISl, VISal, & VISam across all depths first, before splitting by depth.
3. Do other stimulus types (e.g. static gratings, natural movies, etc) have different predictive ability across layers or areas?

# 3. Is behavior itself predictive of transgenic line?
Ok, so clearly neural activity is different in our different transgenic lines / cortical layers. That is why we did experiments on these animals in the first place! Is it possible though that our genetic manipulations also altered the behavior of the animal? Can we *just* look at the behavior of an animal to say something about which transgenic line it is a part of? Remember, all of these lines are created from the same background animal, C57. So, the answer should be no? 

In [ ]:
#Let's look at the running behavior of 2 cre lines, both of which target layer 5 excitatory neurons in cortex! 
cre_list = ['Tlx3-Cre_PL56','Fezf2-CreER']
experiments_sub_df = experiments_df.loc[(experiments_df['cre_line'].isin(cre_list)), ['experiment_container_id', 'targeted_structure','imaging_depth', 'cre_line','reporter_line', 'session_type']]
experiments_sub_df.head(10)

In [ ]:
## EXERCISE: Plot the running speed of a few example sessions, and compare across the two cre lines. Do you see any differences?
# What features could you calculate from the running speed time series, and how would you compare across the two cre lines?
# Use this code snippet to get the running speed time series for a given experiment,
#   ds = boc.get_ophys_experiment_data(exp_id)
#   speed, ts = ds.get_running_speed()


In [ ]:
## Exercise: Compute some simple statistics from the running speed time series, such as mean speed, variance of speed, fraction of time spent running. Do you see any differences in these features across the two cre lines?


In [ ]:
## This one dimensional variable of running speed contains a lot of information about the behavior of the animal during the experiment. 
# We can compute various features from this time series, such as mean speed, variance of speed, fraction of time spent running, number of running bouts per minute, average duration of running bouts, etc. 
# We can then compare these features across the two cre lines to see if there are any differences in their running behavior.
# I ran this function across all ~1400 experiments, so we have a pre-computed table of running features for all experiments, similar to the table we used for the drifting grating metrics above. 
# You can use this table to explore how running behavior varies across different cre lines

def compute_running_features(dataset, speed_threshold_cm_s=2.0, min_bout_duration_s=0.5):
    speed, ts = dataset.get_running_speed()
    valid = np.isfinite(speed) & np.isfinite(ts) & (speed >= -5) & (speed <= 100)
    if valid.sum() < 20:
        return None

    s = speed[valid]
    t = ts[valid]
    dt = float(np.nanmedian(np.diff(t))) if len(t) > 1 else np.nan

    running = s > speed_threshold_cm_s
    frac_running = float(np.mean(running))

    transitions = np.diff(running.astype(int), prepend=0)
    starts = np.where(transitions == 1)[0]
    ends = np.where(transitions == -1)[0] - 1
    if len(running) and running[-1]:
        ends = np.append(ends, len(running) - 1)

    n_pairs = min(len(starts), len(ends))
    starts = starts[:n_pairs]
    ends = ends[:n_pairs]

    n_bouts = n_pairs
    if n_pairs > 0 and np.isfinite(dt):
        bout_durations = (ends - starts + 1) * dt
        keep_bouts = bout_durations >= float(min_bout_duration_s)
        bout_durations = bout_durations[keep_bouts]
        n_bouts = int(keep_bouts.sum())
    else:
        bout_durations = np.array([], dtype=float)

    duration_min = ((t[-1] - t[0]) / 60.0) if len(t) > 1 else np.nan
    bouts_per_min = float(n_bouts / duration_min) if np.isfinite(duration_min) and duration_min > 0 else np.nan
    return {
        "mean_speed": float(np.nanmean(s)),
        "var_speed": float(np.nanvar(s)),
        "max_speed": float(np.nanpercentile(s,99)),
        "frac_running": frac_running,
        "bouts_per_min": bouts_per_min,
        "mean_bout_dur_s": float(np.nanmean(bout_durations)) if len(bout_durations) else 0.0,
        "median_bout_dur_s": float(np.nanmedian(bout_durations)) if len(bout_durations) else 0.0,
        "max_bout_dur_s": float(np.nanmax(bout_durations)) if len(bout_durations) else 0.0
    }

features_csv = Path("running_features_by_experiment.csv")
running_metrics = pd.read_csv(features_csv)
running_metrics.head()

In [ ]:
## EXERCISE: Using  the running_metrics table, can you identify which features differ across the different cre lines?
# We can try using ANOVA to compute the effect size of each feature across the different cre lines, and then visualize the top features by class. 
# We can also create a heatmap of the z-scored locomotion statistics across classes to see if there are any patterns in the data.
# Estimate feature-wise class effect size (eta^2 from one-way ANOVA decomposition)


## 3.1 Can we classify transgenic line using behavioral features?

In [ ]:
## 3.1 Are locomotion features linearly separable by transgenic line?
# Basic QC of per-experiment running features
features = [
    "mean_speed", "var_speed", "max_speed", "frac_running", "bouts_per_min",
    "mean_bout_dur_s", "median_bout_dur_s", "max_bout_dur_s",
]

##This time we will be looking at the running behavior of the animals during the experiments, 
# rather than the visual response properties of the recorded cells.
#
# We can look at the same cre_lines_of_interest as above, or look at all cre lines in the running_metrics table, 
# but for now let's focus on the same 8 lines we looked at above.
decoder_df = running_metrics.loc[running_metrics.cre_line.isin(cre_lines_of_interest)]
decoder_df = decoder_df.replace([np.inf, -np.inf], np.nan)

print(f"Experiments with labels: {len(decoder_df)}")
print("Class counts:")
print(decoder_df["cre_line"].value_counts().head(20))

decoder_df[features].describe().T

In [ ]:
## Use the same approach you used above to decode layer labels from drifting grating metrics, 
# but this time try to decode cre line labels from running behavior features ! !
target = 'cre_line'
X = decoder_df[features].to_numpy()
y = decoder_df[target].to_numpy()
print(f"Samples: {len(decoder_df)}, features: {X.shape[1]}, classes: {np.unique(y).size}")

In [ ]:
## Can you decode transgenic lines with just behavior?! Do we even need to record from the brain?
# What does this mean ? ! 
#
## MORE PROJECT IDEAS:
# 1) Can we combine the neural response features and the running behavior features to improve decoding accuracy?
# -> -> This is a bit tricky because the neural features were calculated on experiment-id level (multiple experiments) whereas the running features were calculated on a per-experiment level. 
# -> -> We would have to decide how to combine these features in a meaningful way, perhaps by averaging the neural features across experiments for each cre line.
# 2) Can we look at more neural features across different stimulus types (e.g. static gratings, natural movies, etc) and see if we can improve decoding accuracy by combining features across stimulus types?